# Pinch Analysis: Crude Preheat Train
This notebook shows how the pinch, utility targets, and graph shapes move when the minimum approach temperature assumption changes.


## Case context
The packaged `crude_preheat_train.json` case represents a small crude-unit preheat train with named hot and cold duties. The workflow is: solve the baseline case, inspect the main graphs, then rerun the case by changing the root-zone `dt_cont_multiplier`, which scales every process and utility stream `dt_cont` value through the zone hierarchy.


In [ ]:
from copy import deepcopy
import json
import numpy as np
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from OpenPinch import PinchProblem
from OpenPinch.resources import copy_sample_case

In [ ]:
workspace = TemporaryDirectory()
work_dir = Path(workspace.name)
case_path = copy_sample_case(
    "crude_preheat_train.json",
    work_dir / "crude_preheat_train.json",
)

In [ ]:
problem = PinchProblem(source=case_path)
problem.target()
summary = problem.summary_frame()
base_row = summary.loc[
    summary["Target"] == "crude_preheat_train/Direct Integration",
    [
        "Hot Utility Target",
        "Cold Utility Target",
        "Heat Recovery",
        "Hot Pinch",
        "Cold Pinch",
    ],
]
print(summary)

## Baseline graphs
Render the baseline composite curve, shifted composite curve, and grand composite curve directly in the notebook so the graph shapes can be read before changing `dt_cont`.


In [ ]:
problem.graph_catalog()

In [ ]:
cc = problem.plot.composite_curve(show=True)

In [ ]:
scc = problem.plot.shifted_composite_curve(show=True)

In [ ]:
bcc = problem.plot.balanced_composite_curve(show=True)

In [ ]:
gcc = problem.plot.grand_composite_curve(show=True)

In [ ]:
nlc = problem.plot.net_load_profiles(show=True)

In [ ]:
multipliers = np.linspace(0.5, 10.0, 26)

problem = PinchProblem(source=case_path, project_name=case_path.stem)

rows = []
for factor in multipliers:
    problem.set_dt_cont_multiplier(factor)
    summary = problem.summary_frame()
    row = summary.loc[summary["Target"] == f"{case_path.stem}/Direct Integration"].iloc[
        0
    ]
    rows.append(
        {
            "dt_cont multiplier": factor,
            "Hot Utility Target": row["Hot Utility Target"],
            "Cold Utility Target": row["Cold Utility Target"],
            "Heat Recovery": row["Heat Recovery"],
            "Hot Pinch": row["Hot Pinch"],
            "Cold Pinch": row["Cold Pinch"],
        }
    )

sensitivity = pd.DataFrame(rows)
sensitivity

In [ ]:
series_colors = {
    "Hot Utility Target": "#c0392b",
    "Cold Utility Target": "#2471a3",
    "Heat Recovery": "#138d75",
    "Hot Pinch": "#d68910",
    "Cold Pinch": "#7d3c98",
}

sensitivity_fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "Utility Targets and Heat Recovery",
        "Pinch Temperatures",
    ),
)

for column in ["Hot Utility Target", "Cold Utility Target", "Heat Recovery"]:
    sensitivity_fig.add_trace(
        go.Scatter(
            x=sensitivity["dt_cont multiplier"],
            y=sensitivity[column],
            mode="lines+markers",
            name=column,
            line={"color": series_colors[column], "width": 3},
            marker={"color": series_colors[column], "size": 7},
        ),
        row=1,
        col=1,
    )

for column in ["Hot Pinch", "Cold Pinch"]:
    sensitivity_fig.add_trace(
        go.Scatter(
            x=sensitivity["dt_cont multiplier"],
            y=sensitivity[column],
            mode="lines+markers",
            name=column,
            line={"color": series_colors[column], "width": 3},
            marker={"color": series_colors[column], "size": 7},
        ),
        row=2,
        col=1,
    )

sensitivity_fig.update_xaxes(title_text="dt_cont multiplier", row=2, col=1)
sensitivity_fig.update_yaxes(title_text="kW or degC", row=1, col=1)
sensitivity_fig.update_yaxes(title_text="degC", row=2, col=1)
sensitivity_fig.update_layout(title="dt_cont sensitivity overview")
# display_plotly(sensitivity_fig, height=700)